# ORB Feature Matching: Essential Validation & Enhancement Study

This notebook evaluates the **4 essential configurations** of the ORB feature matching pipeline on the **139 validation PCB images** (`data/dataset_split.csv`).

### Enhancement Strategy:
1. **Baseline Failure**: Drawing fixed-radius bounding boxes per individual keypoint creates over 63,000 overlapping boxes across the validation set.
2. **Point Clustering Mask (`merge_points=True`)**: Renders anomaly keypoint coordinates onto a binary mask and merges them via contour extraction.
3. **Radius & Filter Tuning**: Balances clustering radius and eliminates isolated single-point false alarms.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import yaml

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'validation' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from algorithms.common import load_image
from algorithms.evaluation import evaluate_boxes, parse_voc_boxes
from algorithms.orb import detect_orb
from algorithms.preprocessing import build_preprocessing_config

plt.rcParams['figure.dpi'] = 110
print(f'Project Root: {PROJECT_ROOT}')

# Smart auto-run fallback: auto-compute benchmark if outputs/metrics CSV is missing
res_csv = PROJECT_ROOT / 'outputs' / 'metrics' / 'essential_validation_comparison.csv'
if not res_csv.exists():
    print('⚠️ Validation metrics CSV not found. Auto-running validation benchmark on 139 validation images...')
    from scripts.run_essential_validation import main as run_benchmark
    run_benchmark()
    print('✅ Benchmark successfully generated!')


## 1. Essential Validation Results (4 Core Configurations)

In [ ]:
df = pd.read_csv(res_csv)
orb_df = df[df['algorithm'] == 'ORB'][['combination_id', 'description', 'precision', 'recall', 'f1_score', 'mean_runtime_ms']]
display(orb_df)


## 2. Visual Comparison of ORB Configurations

In [ ]:
plt.figure(figsize=(9, 4))
plt.barh(orb_df['combination_id'], orb_df['f1_score'], color='#9b59b6')
plt.xlabel('Validation F1-Score (IoU >= 0.50)')
plt.title('ORB Feature Matching: F1-Score across 4 Essential Configurations')
plt.xlim(0, 0.015)
for i, v in enumerate(orb_df['f1_score']):
    plt.text(v + 0.0003, i, f'{v:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()
